# Activation-LQR (A-LQR) on Cosmos-Policy-LIBERO-Predict2-2B — closed-loop rollout

Closed-loop A-LQR steering for the Cosmos-Policy LIBERO action model, run inside a
real LIBERO simulator rollout (not a single inference).

- **Cosmos-Policy** DiT (28 blocks, block output `(B, T_p, H_p, W_p, D)` -- no padded
  sequence dim) and the EDM-style sampler with CFG=cond+uncond per step.
- **Per-partition V** (one V pooled across the SVD's selected timesteps), so
  `V_for(layer_idx, t_id)` keys on `(p_idx, t_id)`.
- **Chain over `selected_timesteps`** rather than the full `sampling_steps`: the
  A_tilde / B_tilde Jacobians only exist at those steps.
- **Closed-loop rollout**: `env.reset() → set_init_state → settle → chunked policy`.
  Each policy call resets `state["pass_idx"]` so the per-denoising-step hooks fire
  correctly within the chunk. Hooks stay registered across the whole rollout (the
  per-chunk `_post_dit_cleanup` is intentionally NOT registered).

## How to iterate from the shell

The notebook reads the following env vars at run time (defaults in parens):

| var          | default                | meaning                                      |
|--------------|------------------------|----------------------------------------------|
| `PROMPT`     | `open the drawer`      | task description passed to the policy        |
| `Q_SCALE`    | `10000.0`              | LQR state cost                               |
| `R_SCALE`    | `75000.0`              | LQR control cost                             |
| `QF_SCALE`   | `1.0`                  | LQR terminal cost                            |
| `LAMBDA`     | `1.0`                  | A-LQR setpoint scaling                       |
| `N_EPISODES` | `10`                   | number of rollouts to record                 |
| `TASK_ID`    | `0`                    | LIBERO task index within `SUITE_NAME`        |
| `SUITE_NAME` | `libero_10`            | LIBERO benchmark suite                       |
| `RESOLUTION` | `256`                  | sim render resolution                        |
| `VIDEO_FPS`  | `30`                   | mp4 fps                                      |
| `SVD_DIR`    | (in cell 4)            | SVD output dir                               |
| `JAC_DIR_ACT`| (in cell 4)            | jacobian output subdir                       |

Run via `run_lqr_cosmos_policy.sh` (it forwards all of the above).

## Outputs

Land under `notebooks/lqr/rollouts/<config_tag>/`, in the same style as
`notebooks/stress_test/rollouts/`:
- `ep{NN}--{SUCCESS|FAILURE}.mp4` — per-episode rollout video (LIBERO agentview)
- `manifest.json`                 — suite/task, prompts, LQR scales, sel_t, seed
- `results.json`                  — per-episode success / env_steps / wall time
- `baseline/`                     — unsteered rollout for the same prompt/seed

## Prerequisites

1. `notebooks/lqr/run_partition_svd_cosmos_policy.sh`
2. `notebooks/lqr/run_jacobians_full.sh`  (defaults to A_tilde only; B_tilde is not
   yet implemented for Cosmos-Policy's EDM 2ab solver, so the chain runs in Wan's
   "degraded" cross-step-zero mode.)


# Section 1 — One-time setup
Run all cells in this section once after kernel start. Do not re-run when changing LQR parameters.

## 1.1 Imports, paths, config

In [ ]:
# --- Cosmos-Policy env setup (must run BEFORE any cosmos_policy import) -----
# Match notebooks/_setup.py and run_partition_svd_cosmos_policy.py:_setup_env(),
# plus MUJOCO_GL setup so the LIBERO OffScreenRenderEnv can render headlessly.
import os, sys
from pathlib import Path

# Headless MuJoCo rendering -- required for LIBERO OffScreenRenderEnv.
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

_HF = os.environ.get("HF_HOME", "/work/nvme/bhde/jhong7/huggingface")
Path(_HF, "hub").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", _HF)
os.environ.setdefault("HF_HUB_CACHE", str(Path(_HF, "hub")))
os.environ.setdefault("TRANSFORMERS_CACHE", str(Path(_HF, "hub")))

if "HF_TOKEN" not in os.environ:
    for _cand in (
        Path(os.path.expanduser("~/.huggingface/token")),
        Path(os.path.expanduser("~/.cache/huggingface/token")),
    ):
        if _cand.exists():
            os.environ["HF_TOKEN"] = _cand.read_text().strip()
            break

_LIBERO = Path(os.environ.get("LIBERO_CONFIG_PATH", "/work/nvme/bhde/jhong7/.libero"))
_LIBERO.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("LIBERO_CONFIG_PATH", str(_LIBERO))

# Make the cosmos-policy repo importable. The repo root is the parent of
# `notebooks/` (so two levels up from this notebook).
_REPO_ROOT = Path.cwd()
while _REPO_ROOT.parent != _REPO_ROOT and not (_REPO_ROOT / "cosmos_policy").is_dir():
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
os.chdir(_REPO_ROOT)
print(f"repo root = {_REPO_ROOT}")


In [ ]:
from pathlib import Path
from collections import OrderedDict
import json, time, gc
import torch
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
_DEVICE_ID = DEVICE.index if DEVICE.type == "cuda" else 0
print(f"DEVICE = {DEVICE}")

# SVD output produced by run_partition_svd_cosmos_policy.sh.
SVD_DIR = Path(os.environ.get(
    "SVD_DIR",
    "/work/nvme/bhde/jhong7/cosmos-policy/directions/svd/"
    "libero10_task00_orig_vs_disturbed_N414_k64_p10_ws8_tsall",
))
# A_tilde output dir produced by run_jacobians_full.sh (named off the prompt
# used at jacobian-collection time).
JAC_DIR_ACT  = SVD_DIR / os.environ.get(
    "JAC_DIR_ACT",
    "A_tilde_full__put_both_the_alphabet_soup_and_the__vjp__vbf16",
)
A_TILDE_FULL = JAC_DIR_ACT / "A_tilde__full.pt"

assert SVD_DIR.exists(), f"SVD_DIR not found: {SVD_DIR}"
assert A_TILDE_FULL.exists(), f"A_TILDE_FULL not found: {A_TILDE_FULL}"

cfg = json.loads((SVD_DIR / "config.json").read_text())
sel_t          = list(cfg["selected_timesteps"])
T_diff         = len(sel_t)                      # number of LQR steps in the chain
sampling_steps = cfg["sampling_steps"]           # total denoising steps the sampler runs
L              = cfg["L"]
r              = cfg["k_target"]
partitions     = [tuple(p) for p in cfg["partitions"]]
print(f"sel_t={sel_t}  T_diff(chain)={T_diff}  sampling_steps={sampling_steps}  "
      f"L={L}  r={r}  partitions={partitions}  D={cfg['D']:,}")


## 1.2 Load $\tilde A$, $\tilde B$, LFS

In [ ]:
raw = torch.load(A_TILDE_FULL, map_location="cpu", weights_only=False)
A_dict     = raw.get("A_tilde", {})
B_dict     = raw.get("B_tilde", {})
JAC_PROMPT = raw.get("prompt", "<unknown>")
print(f"A_tilde dict: {len(A_dict)} entries (expected {T_diff*(L-1)})  "
      f"B_tilde dict: {len(B_dict)} entries  prompt={JAC_PROMPT!r}")

# A_tilde keyed by (actual step in trajectory, l_in); fold to (sel_idx, l_in).
sel_idx_of = {t: i for i, t in enumerate(sel_t)}
A_tilde = torch.zeros(T_diff, L - 1, r, r, dtype=torch.float32)
for (t, l_in), Atl in A_dict.items():
    if t in sel_idx_of:
        A_tilde[sel_idx_of[t], l_in] = Atl.float()

HAVE_B = len(B_dict) > 0
if HAVE_B:
    B_tilde = torch.zeros(T_diff - 1, r, r, dtype=torch.float32)
    for (t,), Bt in B_dict.items():
        if t in sel_idx_of and sel_idx_of[t] < T_diff - 1:
            B_tilde[sel_idx_of[t]] = Bt.float()
else:
    print("[note] B_tilde empty (cosmos-policy default); chained Riccati degrades "
          "to per-step (zero cross-step A matrices).")
    B_tilde = torch.zeros(max(T_diff - 1, 0), r, r, dtype=torch.float32)

# LFS from c_means. With per-(partition, timestep) SVD, c_means is now
# (L, T_sel, k_target) -- one contrastive vector per (layer, timestep).
summary = torch.load(SVD_DIR / "svd_summary.pt", map_location="cpu", weights_only=False)
c_means = summary["c_means"].float()  # (L, T_sel, k)
assert c_means.dim() == 3, (
    f"expected c_means shape (L, T_sel, k); got {tuple(c_means.shape)}"
)
tilde_mu = c_means.norm(dim=-1)                            # (L, T_sel)
tilde_v  = c_means / tilde_mu.unsqueeze(-1).clamp(min=1e-12)  # (L, T_sel, k)
layer_to_part = list(summary["layer_to_part"])
print(f"A_tilde {tuple(A_tilde.shape)}  B_tilde {tuple(B_tilde.shape)}  "
      f"tilde_v {tuple(tilde_v.shape)}  tilde_mu {tuple(tilde_mu.shape)}")

finite_A = A_tilde.flatten(2).isfinite().all().item()
finite_B = B_tilde.numel() == 0 or B_tilde.flatten(1).isfinite().all().item()
A_norm = A_tilde.flatten(2).norm(dim=-1).mean().item()
B_norm = B_tilde.flatten(1).norm(dim=-1).mean().item() if B_tilde.numel() else 0.0
print(f"finite: A={finite_A} B={finite_B}  ||A||_F mean={A_norm:.3f}  "
      f"||B||_F mean={B_norm:.3f}")


## 1.3 Cosmos-Policy loader

In [ ]:
from cosmos_policy.experiments.robot.libero.run_libero_eval import (
    PolicyEvalConfig,
)
from cosmos_policy.experiments.robot.cosmos_utils import (
    COSMOS_IMAGE_SIZE,
    get_action,
    get_model,
    init_t5_text_embeddings_cache,
    load_dataset_stats,
)

assert DEVICE.type == "cuda", f"Cosmos-Policy requires CUDA; DEVICE={DEVICE}"
_DEVICE_ID = DEVICE.index if DEVICE.index is not None else 0

# Rollout knobs (defaults; overridden by env vars in the shell launcher). Set
# them here so the eval_cfg below can read SUITE_NAME for task_suite_name.
PROMPT     = os.environ.get("PROMPT", "open the drawer")
SUITE_NAME = os.environ.get("SUITE_NAME", "libero_10")
TASK_ID    = int(os.environ.get("TASK_ID", "0"))
N_EPISODES = int(os.environ.get("N_EPISODES", "10"))
RESOLUTION = int(os.environ.get("RESOLUTION", "256"))
VIDEO_FPS  = int(os.environ.get("VIDEO_FPS", "30"))
print(f"PROMPT     = {PROMPT!r}")
print(f"SUITE_NAME = {SUITE_NAME}    TASK_ID = {TASK_ID}    N_EPISODES = {N_EPISODES}")

ckpt_path = cfg.get("ckpt_path", "nvidia/Cosmos-Policy-LIBERO-Predict2-2B")
config_name = cfg.get("config_name",
                       "cosmos_predict2_2b_480p_libero__inference_only")
config_file = "cosmos_policy/config/config.py"
dataset_stats_path = f"{ckpt_path}/libero_dataset_statistics.json"
t5_cache_path = f"{ckpt_path}/libero_t5_embeddings.pkl"

# Real LIBERO rollouts: use the production flag set (matching
# `notebooks/lqr/collect_policy_inputs.ipynb`, which generated the observations
# the SVD/jacobians were fit against). The model sees flipped + JPEG-compressed
# images and normalized proprio, and env.step receives unnormalized actions.
# The SVD code itself sets all five to False because its inputs.npz was already
# pre-processed at capture time -- the model's effective inputs are the same
# in both cases.
eval_cfg = PolicyEvalConfig(
    config=config_name,
    ckpt_path=ckpt_path,
    config_file=config_file,
    dataset_stats_path=dataset_stats_path,
    t5_text_embeddings_path=t5_cache_path,
    use_wrist_image=True,
    use_proprio=True,
    normalize_proprio=True,
    unnormalize_actions=True,
    chunk_size=16,
    num_open_loop_steps=16,
    trained_with_image_aug=True,
    use_jpeg_compression=True,
    flip_images=True,
    num_denoising_steps_action=sampling_steps,
    num_denoising_steps_future_state=1,
    num_denoising_steps_value=1,
    task_suite_name=SUITE_NAME,
)

print(f"loading dataset stats + T5 cache + Cosmos-Policy model on cuda:{_DEVICE_ID} ...")
dataset_stats = load_dataset_stats(eval_cfg.dataset_stats_path)
init_t5_text_embeddings_cache(eval_cfg.t5_text_embeddings_path, worker_id=0)
model, _ = get_model(eval_cfg)
n_blocks = len(model.net.blocks)
print(f"loaded; {n_blocks} DiT blocks (expected {L})")
assert n_blocks == L

gc.collect()
torch.cuda.empty_cache()
free_gb, total_gb = (x / 1e9 for x in torch.cuda.mem_get_info(_DEVICE_ID))
print(f"after loader cleanup: GPU {free_gb:.1f} / {total_gb:.1f} GB free")


## 1.4 Per-partition V cache

In [ ]:
# Token-space dims (must match the SVD config).
P_tok           = cfg["P_tok"]
D               = cfg["D"]
D_flat          = cfg["D_flat"]
T_p_denoise     = cfg["T_p_denoise"]
denoise_t_start = cfg["denoise_t_start"]
denoise_t_end   = cfg["denoise_t_end"]
H_p             = cfg["H_p"]
W_p             = cfg["W_p"]
print(f"T_p_denoise={T_p_denoise}  denoise=[{denoise_t_start}..{denoise_t_end-1}]  "
      f"H_p={H_p}  W_p={W_p}  D={D}  P_tok={P_tok}  D_flat={D_flat:,}")

# Cosmos-Policy V is now per (partition, timestep). With 3 partitions × T_sel
# timesteps and k=64 on D_flat~2.0M, each V is ~512 MB fp32 / ~257 MB bf16,
# total ~3.85 GB bf16 across all (partition, t) tiles. We cache by
# (p_idx, t_id); MAX_GPU_PARTITIONS is interpreted as the max number of
# (partition, timestep) tiles resident on V_DEVICE.
V_DEVICE = DEVICE
V_DTYPE  = torch.bfloat16
MAX_GPU_PARTITIONS = len(partitions) * len(sel_t)  # all tiles fit on GH200

_V_CPU_CACHE: dict = {}
_V_GPU_CACHE: "OrderedDict" = OrderedDict()
_swap_stats = {"cpu_loads": 0, "gpu_swaps": 0, "gpu_hits": 0, "cpu_to_gpu_s": 0.0}


def _v_file(p_idx: int, t_id: int) -> Path:
    a, b = partitions[p_idx]
    pattern = f"V_part{p_idx}_layers{a}-{b}_t{t_id}_k*.pt"
    matches = sorted(SVD_DIR.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"no V file matching {pattern} in {SVD_DIR}")
    preferred = [m for m in matches if m.name.endswith(f"_k{cfg['k_target']}.pt")]
    return (preferred or matches)[0]


def _load_V_cpu(p_idx: int, t_id: int) -> torch.Tensor:
    key = (p_idx, t_id)
    V = _V_CPU_CACHE.get(key)
    if V is not None:
        return V
    fp = _v_file(p_idx, t_id)
    print(f"  disk-load {fp.name} ({fp.stat().st_size/1e9:.2f} GB) -> CPU {V_DTYPE} ...")
    t0 = time.time()
    raw = torch.load(fp, map_location="cpu", weights_only=False)["V"]
    V = raw.to(dtype=V_DTYPE).contiguous()
    del raw
    _V_CPU_CACHE[key] = V
    _swap_stats["cpu_loads"] += 1
    print(f"    CPU resident (p={p_idx}, t={t_id}): {tuple(V.shape)}  "
          f"~{V.element_size()*V.numel()/1e9:.2f} GB  ({time.time()-t0:.1f}s)")
    return V


def V_for(layer_idx: int, t_id: int) -> torch.Tensor:
    """Return the V matrix for the (partition, timestep) containing layer
    `layer_idx` at the actual diffusion step `t_id`."""
    p_idx = layer_to_part[layer_idx]
    key = (p_idx, t_id)
    V = _V_GPU_CACHE.get(key)
    if V is not None:
        _V_GPU_CACHE.move_to_end(key)
        _swap_stats["gpu_hits"] += 1
        return V
    V_cpu = _load_V_cpu(p_idx, t_id)
    if V_DEVICE.type == "cpu":
        _V_GPU_CACHE[key] = V_cpu
        return V_cpu
    while len(_V_GPU_CACHE) >= MAX_GPU_PARTITIONS:
        _, ev_V = _V_GPU_CACHE.popitem(last=False)
        del ev_V
        torch.cuda.empty_cache()
        _swap_stats["gpu_swaps"] += 1
    t0 = time.time()
    V = V_cpu.to(device=V_DEVICE, non_blocking=False).contiguous()
    if V_DEVICE.type == "cuda":
        torch.cuda.synchronize(V_DEVICE)
    _swap_stats["cpu_to_gpu_s"] += time.time() - t0
    _V_GPU_CACHE[key] = V
    return V


print(f"CPU pre-load of all {len(partitions)} × {len(sel_t)} V (partition, t) tiles ...")
for p_idx in range(len(partitions)):
    for t_id in sel_t:
        _load_V_cpu(p_idx, t_id)
V_for(partitions[0][0], sel_t[0])
print(f"CPU resident: {len(_V_CPU_CACHE)} tiles   "
      f"{V_DEVICE} resident (MAX={MAX_GPU_PARTITIONS}): "
      f"{sorted(_V_GPU_CACHE.keys())[:5]}{'...' if len(_V_GPU_CACHE) > 5 else ''}")
if V_DEVICE.type == "cuda":
    free_gb, total_gb = (x / 1e9 for x in torch.cuda.mem_get_info(_DEVICE_ID))
    print(f"after V warm: GPU {free_gb:.1f} / {total_gb:.1f} GB free")


## 1.5 Hook definitions

Defines the closed-loop A-LQR hook closures. The closures read `_lqr`, `LAMBDA`, and `state` from module globals at *call* time, not definition time, so this cell can run before those globals exist. Section 3's generate cell populates `_lqr` and resets `state` on every run.

Cosmos-Policy fires the DiT twice per denoising step (cond + uncond, always; see `policy_video2world_model.x0_fn`). Only the cond pass is steered: `pass_idx % 2 == 0`, and `step = pass_idx // 2`. Within-step `make_intra_hook(l_in)` is a post-hook on `block[l_in+1]` that projects with `V_for(l_in)`, computes the LQR control, lifts via `V_for(l_in+1)`, and adds to the block output. Cross-step `cross_step_compute` (post on `block[L-1]`) stashes the r-dim u_tilde; `cross_step_apply` (post on `block[0]`) lifts it via `V_for(0)` at the next selected step. With Cosmos-Policy's V being per-partition, the cache key reduces to `p_idx` alone.

In [ ]:
# Mutable globals; populated/reset by the Section 3 generate cell.
state = {"pass_idx": -1, "u_step_pending": None}
in_ad = {"flag": False}
u_norm_log: list = []
_RUN_T0 = None

# Cosmos-Policy `get_action` codepath runs ONE forward per denoising step
# (no CFG -- is_negative_prompt=False in generate_samples_from_batch, so
# uncondition is None in the sampler). pass_idx == step directly.
PASSES_PER_STEP = 1


def _split_context(ctx):
    """context_input may be a tensor (text-only) or (text, img_context) tuple."""
    if isinstance(ctx, tuple):
        return ctx[0], ctx[1:]
    return ctx, None


def _is_selected_step(pass_idx: int):
    """Return (step, sel_idx) if this pass is a selected denoising step; else
    (None, None)."""
    if pass_idx < 0:
        return None, None
    step = pass_idx  # PASSES_PER_STEP == 1
    if step not in sel_idx_of:
        return None, None
    return step, sel_idx_of[step]


def _pass_tick(_block, _args):
    if not in_ad["flag"]:
        state["pass_idx"] += 1
        if state["pass_idx"] == 0:
            print(f"  [t=+{time.time()-_RUN_T0:.1f}s] pass 0 (step 0) - "
                  f"block 0 about to run", flush=True)
    return None


def make_intra_hook(l_in: int):
    """Post-hook on block[l_in+1]: project current activation, compute u_tilde
    via LQR, lift back, and add to the block output (denoising slots only)."""
    def hook(block, args, output):
        if in_ad["flag"]:
            return None
        step, sel = _is_selected_step(state["pass_idx"])
        if step is None:
            return None
        # block output is (B, T_p, H_p, W_p, D); restrict to denoising slots.
        z_full = (
            output[0, denoise_t_start:denoise_t_end, :, :, :]
            .detach().reshape(-1)
        )
        z_dt   = output.dtype
        V_in = V_for(l_in, step)
        in_ad["flag"] = True
        try:
            x_proj  = (z_full.to(V_in.dtype) @ V_in).float()
            v_fp    = _lqr["v"][l_in, sel]
            mu_fp   = _lqr["mu"][l_in, sel]
            K_fp    = _lqr["K_intra"][sel, l_in]
            alpha   = LAMBDA * mu_fp - v_fp @ x_proj
            u_tilde = K_fp @ (alpha * v_fp)
            u_norm_log.append((sel, l_in, float(u_tilde.norm()), float(alpha)))
        finally:
            in_ad["flag"] = False
        del V_in

        V_out = V_for(l_in + 1, step)
        in_ad["flag"] = True
        try:
            u_full = V_out @ u_tilde.to(V_out.dtype)
        finally:
            in_ad["flag"] = False
        u_add = u_full.to(z_dt).reshape(T_p_denoise, H_p, W_p, D)
        output[0, denoise_t_start:denoise_t_end, :, :, :] = (
            output[0, denoise_t_start:denoise_t_end, :, :, :] + u_add
        )
        return output
    return hook


def cross_step_compute(block, args, output):
    """Post-hook on block[L-1]; stashes u_tilde for the next selected step."""
    if in_ad["flag"]:
        return None
    step, sel = _is_selected_step(state["pass_idx"])
    if step is None or sel >= T_diff - 1:
        return None
    z_full = (
        output[0, denoise_t_start:denoise_t_end, :, :, :]
        .detach().reshape(-1)
    )
    V_in = V_for(L - 1, step)
    in_ad["flag"] = True
    try:
        x_proj  = (z_full.to(V_in.dtype) @ V_in).float()
        v_fp    = _lqr["v"][L - 1, sel]
        mu_fp   = _lqr["mu"][L - 1, sel]
        K_fp    = _lqr["K_step"][sel]
        alpha   = LAMBDA * mu_fp - v_fp @ x_proj
        u_tilde = K_fp @ (alpha * v_fp)
        u_norm_log.append((sel, -1, float(u_tilde.norm()), float(alpha)))
        state["u_step_pending"] = {"src_sel": sel, "u_tilde": u_tilde.detach()}
    finally:
        in_ad["flag"] = False
    return None


def cross_step_apply(block, args, output):
    """Post-hook on block[0]: if we are at the next selected step and a u is
    pending, lift it via V_for(0, step) and add to the denoising-slot output."""
    if in_ad["flag"]:
        return None
    step, sel = _is_selected_step(state["pass_idx"])
    pending = state["u_step_pending"]
    if step is None or pending is None or sel == 0 or pending["src_sel"] != sel - 1:
        return None
    u_tilde = pending["u_tilde"]
    V_dest = V_for(0, step)
    in_ad["flag"] = True
    try:
        u_full = V_dest @ u_tilde.to(V_dest.dtype)
    finally:
        in_ad["flag"] = False
    u_add = u_full.to(output.dtype).reshape(T_p_denoise, H_p, W_p, D)
    output[0, denoise_t_start:denoise_t_end, :, :, :] = (
        output[0, denoise_t_start:denoise_t_end, :, :, :] + u_add
    )
    state["u_step_pending"] = None
    return output


def _post_dit_cleanup(block, args, output):
    """After the last selected denoising step's block[L-1], drop V cache and
    LQR push so VAE decode + future-image pipeline run with headroom."""
    step = state["pass_idx"]
    if step != sampling_steps - 1:
        return None
    bytes_freed = sum(V.element_size()*V.numel() for V in _V_GPU_CACHE.values())
    if "_lqr" in globals():
        bytes_freed += sum(t.element_size()*t.numel() for t in _lqr.values())
    n_tiles = len(_V_GPU_CACHE)
    _V_GPU_CACHE.clear()
    if "_lqr" in globals():
        _lqr.clear()
    torch.cuda.empty_cache()
    print(f"  [t=+{time.time()-_RUN_T0:.1f}s] DiT done at last step (pass_idx="
          f"{state['pass_idx']}); freed {n_tiles} V (partition, t) tile(s) "
          f"+ LQR = {bytes_freed/1e9:.1f} GB", flush=True)
    return None


print("hooks defined: _pass_tick, make_intra_hook, cross_step_compute, "
      "cross_step_apply, _post_dit_cleanup")


# Section 2 — Tunable LQR cost
Re-run cells in this section when you change `Q_SCALE`, `R_SCALE`, or `QF_SCALE`. After re-running, also re-run Section 3 (which pushes the new K matrices to GPU).

## 2.1 LQR cost hyperparameters + chained Riccati

In [ ]:
# --- LQR cost hyperparameters (override via env vars). ----------------------
Q_SCALE  = float(os.environ.get("Q_SCALE",  "10000.0"))
R_SCALE  = float(os.environ.get("R_SCALE",  "75000.0"))
QF_SCALE = float(os.environ.get("QF_SCALE", "1.0"))
print(f"Q_SCALE={Q_SCALE:g}  R_SCALE={R_SCALE:g}  QF_SCALE={QF_SCALE:g}")

K_total = T_diff * L - 1 if T_diff > 0 else 0
print(f"chain length: T*L - 1 = {K_total} transitions  "
      f"({T_diff*(L-1)} within-step + {max(T_diff-1, 0)} across-step)")

_LQR_DEVICE = DEVICE
_LQR_DTYPE  = torch.float64
I_r = torch.eye(r, dtype=_LQR_DTYPE, device=_LQR_DEVICE)
Q_chain   = (Q_SCALE  * I_r).expand(K_total, r, r).contiguous()
R_chain   = (R_SCALE  * I_r).expand(K_total, r, r).contiguous()
S_T_chain = (QF_SCALE * I_r).contiguous()

A_tilde_dev = A_tilde.to(device=_LQR_DEVICE, dtype=_LQR_DTYPE)
B_tilde_dev = B_tilde.to(device=_LQR_DEVICE, dtype=_LQR_DTYPE)

A_chain = torch.zeros(K_total, r, r, dtype=_LQR_DTYPE, device=_LQR_DEVICE)
for t in range(T_diff):
    for l in range(L - 1):
        A_chain[t * L + l] = A_tilde_dev[t, l]
    if t < T_diff - 1 and B_tilde_dev.numel():
        A_chain[t * L + (L - 1)] = B_tilde_dev[t]


def lqr_no_B(A_t, Q, R, S_T):
    Tn, n, _ = A_t.shape
    S = torch.zeros(Tn + 1, n, n, dtype=A_t.dtype, device=A_t.device)
    K = torch.zeros(Tn,     n, n, dtype=A_t.dtype, device=A_t.device)
    S[Tn] = S_T
    for k in reversed(range(Tn)):
        Ak = A_t[k]
        P = S[k + 1] + R[k]
        F = S[k + 1] @ Ak
        G = Q[k] + Ak.transpose(-2, -1) @ S[k + 1] @ Ak
        Kk = torch.linalg.solve(P, F)
        K[k] = Kk
        Snew = G - F.transpose(-2, -1) @ Kk
        S[k] = 0.5 * (Snew + Snew.transpose(-2, -1))
    return K


t0 = time.time()
K_chain = lqr_no_B(A_chain, Q_chain, R_chain, S_T_chain)
if _LQR_DEVICE.type == "cuda":
    torch.cuda.synchronize(_LQR_DEVICE)
K_chain = K_chain.float().cpu()

K_intra = torch.zeros(T_diff, L - 1, r, r, dtype=torch.float32)
K_step  = torch.zeros(max(T_diff - 1, 0), r, r, dtype=torch.float32)
for t in range(T_diff):
    for l in range(L - 1):
        K_intra[t, l] = K_chain[t * L + l]
    if t < T_diff - 1:
        K_step[t] = K_chain[t * L + (L - 1)]
print(f"chained Riccati ({K_total} transitions, r={r}) done in {time.time()-t0:.2f}s; "
      f"K_intra {tuple(K_intra.shape)}  K_step {tuple(K_step.shape)}")

del A_chain, Q_chain, R_chain, S_T_chain, I_r, A_tilde_dev, B_tilde_dev
if _LQR_DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 2.2 K diagnostic plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
K_intra_norm = K_intra.flatten(2).norm(dim=-1).numpy()

im0 = axes[0].imshow(K_intra_norm, aspect="auto", origin="lower", cmap="viridis")
axes[0].set_xlabel(r"within-step layer transition $l_{\rm in}$")
axes[0].set_ylabel(r"selected step index $t_{\rm sel}$")
axes[0].set_title(r"$\Vert K^{\rm chain}_{t,l}\Vert_F$ (within-step)")
for s, e in partitions[:-1]:
    axes[0].axvline(e + 0.5, color="white", lw=0.8, alpha=0.7)
plt.colorbar(im0, ax=axes[0])

if K_step.numel() > 0:
    K_step_norm = K_step.flatten(1).norm(dim=-1).numpy()
    axes[1].plot(range(T_diff - 1), K_step_norm, "o-", color="C3")
    axes[1].set_title(r"Cross-step gain magnitude vs $t_{\rm sel}$")
else:
    axes[1].text(0.5, 0.5, "no cross-step transitions\n(T_sel <= 1 or B_tilde absent)",
                  ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_title("Cross-step gain magnitude")
axes[1].set_xlabel(r"selected step $t_{\rm sel}$")
axes[1].set_ylabel(r"$\Vert K\Vert_F$")
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"||K_intra||_F: mean={K_intra_norm.mean():.4e}  max={K_intra_norm.max():.4e}")
if K_step.numel() > 0:
    print(f"||K_step ||_F: mean={K_step_norm.mean():.4e}  max={K_step_norm.max():.4e}")


# Section 3 — Generate
Re-run these cells whenever `PROMPT` or `LAMBDA` changes (or after re-running Section 2). The generate cell handles all per-run setup: state reset, push of fresh K/v/mu to GPU, V cache warm, hook registration, generate, save.

## 3.1 Prompt + rollout knobs

In [ ]:
# Rollout knobs were already read in cell 8 (so the eval_cfg can pick up
# SUITE_NAME). Restate them here for visibility; override via env vars.
PROMPT     = os.environ.get("PROMPT", PROMPT)
SUITE_NAME = os.environ.get("SUITE_NAME", SUITE_NAME)
TASK_ID    = int(os.environ.get("TASK_ID", TASK_ID))
N_EPISODES = int(os.environ.get("N_EPISODES", N_EPISODES))
RESOLUTION = int(os.environ.get("RESOLUTION", RESOLUTION))
VIDEO_FPS  = int(os.environ.get("VIDEO_FPS", VIDEO_FPS))
print(f"PROMPT     = {PROMPT!r}")
print(f"SUITE_NAME = {SUITE_NAME}    TASK_ID = {TASK_ID}")
print(f"N_EPISODES = {N_EPISODES}    RESOLUTION = {RESOLUTION}    VIDEO_FPS = {VIDEO_FPS}")

# Novel prompts that aren't in the T5 cache trigger an on-the-fly T5-11B load +
# embed on the first rollout that uses them (~30-60 s). Subsequent rollouts hit
# the persistent cache.


## 3.2 Steered closed-loop rollout

In [ ]:
LAMBDA = float(os.environ.get("LAMBDA", "1.0"))
SEED   = int(cfg.get("seed", 42))
print(f"LAMBDA={LAMBDA}    SEED={SEED}")

# --- Reset per-run state. ----------------------------------------------------
state["pass_idx"]       = -1
state["u_step_pending"] = None
in_ad["flag"]           = False
u_norm_log.clear()

# --- Push current K_intra / K_step / v / mu to V_DEVICE. ---------------------
_lqr = {
    "K_intra": K_intra.to(device=V_DEVICE, dtype=torch.float32),
    "K_step":  K_step.to(device=V_DEVICE, dtype=torch.float32) if K_step.numel() else K_step,
    "v":       tilde_v.to(device=V_DEVICE, dtype=torch.float32),
    "mu":      tilde_mu.to(device=V_DEVICE, dtype=torch.float32),
}
total_mb = sum(t.element_size()*t.numel() for t in _lqr.values()) / 1e6
print(f"_lqr pushed to {V_DEVICE} ({total_mb:.1f} MB)")

# --- Re-warm V cache if cleared by a previous run. ---------------------------
if not _V_GPU_CACHE:
    V_for(partitions[0][0], sel_t[0])
print(f"V resident on {V_DEVICE}: {len(_V_GPU_CACHE)} tile(s)")

# --- LIBERO env + init states. -----------------------------------------------
from libero.libero import benchmark
from cosmos_policy.experiments.robot.libero.libero_utils import (
    get_libero_env, get_libero_dummy_action,
)
from cosmos_policy.experiments.robot.libero.run_libero_eval import (
    prepare_observation, TASK_MAX_STEPS,
)
from collections import deque
import imageio

task_suite = benchmark.get_benchmark_dict()[SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
init_states = task_suite.get_task_init_states(TASK_ID)
assert N_EPISODES <= init_states.shape[0], (
    f"only {init_states.shape[0]} init states available; requested {N_EPISODES}"
)
env, task_desc_libero = get_libero_env(task, "cosmos", resolution=RESOLUTION)
max_env_steps = TASK_MAX_STEPS[SUITE_NAME]
print(f"libero env ready ({SUITE_NAME} task {TASK_ID:02d})  "
      f"task_desc_libero={task_desc_libero!r}  max_steps={max_env_steps}")
if PROMPT != task_desc_libero:
    print(f"  note: policy prompt differs from LIBERO task description; success is\n"
          f"        tracked against the LIBERO predicate, NOT the prompt.")

# --- Output dir. -------------------------------------------------------------
# _slug truncates PROMPT to 24 chars, so prompts sharing that prefix produce
# the same base CONFIG_TAG and would silently overwrite each other. RUN_TAG
# (env var from the shell launcher) disambiguates: when the base dir already
# exists from a prior run, append `__{RUN_TAG}` instead of overwriting.
def _slug(s, n=24):
    return s[:n].strip().lower().replace(" ", "_").replace("/", "_").replace(".", "_")

_BASE_CONFIG_TAG = (
    f"{SUITE_NAME}__task{TASK_ID:02d}__lqr__"
    f"lam{LAMBDA:.2f}_q{Q_SCALE:g}_r{R_SCALE:g}_qf{QF_SCALE:g}__"
    f"{_slug(PROMPT)}"
)
RUN_TAG = os.environ.get("RUN_TAG", "").strip()
_ROLLOUTS_ROOT = Path("notebooks/lqr/rollouts")
CONFIG_TAG = _BASE_CONFIG_TAG
if (_ROLLOUTS_ROOT / _BASE_CONFIG_TAG).exists():
    if RUN_TAG:
        CONFIG_TAG = f"{_BASE_CONFIG_TAG}__{RUN_TAG}"
        print(f"[output dir] {_BASE_CONFIG_TAG!r} already exists; "
              f"appending RUN_TAG={RUN_TAG!r}")
    else:
        print(f"[output dir] WARNING: {_BASE_CONFIG_TAG!r} already exists "
              f"and RUN_TAG is empty -- previous results will be overwritten. "
              f"Set RUN_TAG in the shell launcher to disambiguate.")
OUT_DIR = _ROLLOUTS_ROOT / CONFIG_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output dir -> {OUT_DIR.resolve()}")

manifest = {
    "suite":             SUITE_NAME,
    "task_id":           TASK_ID,
    "task_desc_libero":  task_desc_libero,
    "policy_prompt":     PROMPT,
    "n_episodes":        N_EPISODES,
    "max_env_steps":     max_env_steps,
    "lambda":            LAMBDA,
    "Q_SCALE":           Q_SCALE,
    "R_SCALE":           R_SCALE,
    "QF_SCALE":          QF_SCALE,
    "seed":              SEED,
    "sel_t":             sel_t,
    "sampling_steps":    sampling_steps,
    "jac_prompt":        JAC_PROMPT,
    "svd_dir":           str(SVD_DIR),
    "resolution":        RESOLUTION,
    "run_tag":           RUN_TAG,
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

# --- Register steering hooks. ------------------------------------------------
# NOTE: we deliberately do NOT register `_post_dit_cleanup` here. That hook
# clears `_V_GPU_CACHE` + `_lqr` after the last denoising step of each chunk,
# which would force a re-warm every chunk during a multi-step rollout. We
# clean up manually at the end of this cell instead.
handles = [
    model.net.blocks[0].register_forward_pre_hook(_pass_tick),
    model.net.blocks[0].register_forward_hook(cross_step_apply),
]
for l_in in range(L - 1):
    handles.append(
        model.net.blocks[l_in + 1].register_forward_hook(make_intra_hook(l_in))
    )
handles.append(model.net.blocks[L - 1].register_forward_hook(cross_step_compute))
print(f"registered {len(handles)} steering hooks (no _post_dit_cleanup)")


def steered_policy_fn(observation, desc):
    """One chunk of A-LQR-steered actions; resets per-chunk hook state first."""
    state["pass_idx"]       = -1
    state["u_step_pending"] = None
    with torch.inference_mode():
        out = get_action(
            eval_cfg, model, dataset_stats, observation, desc,
            seed=SEED,
            randomize_seed=False,
            num_denoising_steps_action=sampling_steps,
            generate_future_state_and_value_in_parallel=True,
        )
    return out["actions"]


def rollout(env, init_state, task_desc, policy_fn, *, episode_idx=0, num_steps_wait=10):
    env.reset()
    obs = env.set_init_state(init_state)
    for _ in range(num_steps_wait):
        obs, _, _, _ = env.step(get_libero_dummy_action(eval_cfg.model_family))
    queue = deque(maxlen=eval_cfg.num_open_loop_steps)
    frames = [obs["agentview_image"].copy()]
    success = False
    t = 0
    while t < max_env_steps:
        if not queue:
            observation = prepare_observation(
                obs, resize_size=COSMOS_IMAGE_SIZE, flip_images=eval_cfg.flip_images,
            )
            actions = policy_fn(observation, task_desc)
            for a in actions[: eval_cfg.num_open_loop_steps]:
                queue.append(np.asarray(a, dtype=np.float32))
        a = queue.popleft()
        obs, _, done, _ = env.step(a.tolist())
        frames.append(obs["agentview_image"].copy())
        if done:
            success = True
            break
        t += 1
    return success, t + num_steps_wait, frames


def save_video(frames, path, fps=VIDEO_FPS):
    writer = imageio.get_writer(path, fps=fps)
    for frame in frames:
        # LIBERO agentview is rendered upside-down; flip for human viewing
        # (matches notebooks/stress_test/01_robot_color.ipynb).
        writer.append_data(np.flipud(frame))
    writer.close()


# --- Run N_EPISODES with the steered policy. ---------------------------------
results = []
_RUN_T0 = time.time()
for ep in range(N_EPISODES):
    t0 = time.time()
    success, env_steps, frames = rollout(
        env, init_states[ep], PROMPT, steered_policy_fn, episode_idx=ep,
    )
    tag = "SUCCESS" if success else "FAILURE"
    mp4 = OUT_DIR / f"ep{ep:02d}--{tag}.mp4"
    save_video(frames, mp4)
    dt = time.time() - t0
    print(f"ep {ep:2d}: {tag:7s}  steps={env_steps:4d}  {dt:6.1f}s  -> {mp4.name}",
          flush=True)
    results.append({
        "episode":     ep,
        "success":     success,
        "env_steps":   env_steps,
        "wall_time_s": dt,
        "video_path":  str(mp4),
    })

(OUT_DIR / "results.json").write_text(json.dumps(results, indent=2))

# --- Tear down hooks; keep env open for the baseline cell. -------------------
for h in handles:
    h.remove()
n_succ = sum(r["success"] for r in results)
print(f"removed {len(handles)} hooks; steered total {time.time()-_RUN_T0:.1f}s; "
      f"V swap stats: hits={_swap_stats['gpu_hits']} swaps={_swap_stats['gpu_swaps']} "
      f"cpu->gpu={_swap_stats['cpu_to_gpu_s']:.1f}s")
print(f"steered: {n_succ}/{len(results)} succeeded ({100*n_succ/len(results):.0f}%)")
print(f"saved -> {OUT_DIR.resolve()}")

# Section 4 — Optional

## 4.1 Unsteered baseline (same prompt + seed, hooks stripped)

Runs the same closed-loop rollout with no steering hooks, into
`<OUT_DIR>/baseline/`. Useful as a side-by-side comparison for the same
`PROMPT` / `SEED` / `TASK_ID`.

In [ ]:
# # Strip any stale hooks; clear V/LQR caches to free GPU for baseline.
# n_hooks_stripped = 0
# for _b in model.net.blocks:
#     n_hooks_stripped += len(_b._forward_pre_hooks) + len(_b._forward_hooks)
#     _b._forward_pre_hooks.clear()
#     _b._forward_hooks.clear()
# if "_V_GPU_CACHE" in globals():
#     _V_GPU_CACHE.clear()
# if "_lqr" in globals():
#     _lqr.clear()
# gc.collect()
# torch.cuda.empty_cache()
# free_gb, total_gb = (x / 1e9 for x in torch.cuda.mem_get_info(_DEVICE_ID))
# print(f"pre-baseline: GPU {free_gb:.1f} / {total_gb:.1f} GB free  "
#       f"(stripped {n_hooks_stripped} hook(s))")

# BASELINE_DIR = OUT_DIR / "baseline"
# BASELINE_DIR.mkdir(parents=True, exist_ok=True)


# def vanilla_policy_fn(observation, desc):
#     with torch.inference_mode():
#         out = get_action(
#             eval_cfg, model, dataset_stats, observation, desc,
#             seed=SEED,
#             randomize_seed=False,
#             num_denoising_steps_action=sampling_steps,
#             generate_future_state_and_value_in_parallel=True,
#         )
#     return out["actions"]


# baseline_results = []
# _BASE_T0 = time.time()
# for ep in range(N_EPISODES):
#     t0 = time.time()
#     success, env_steps, frames = rollout(
#         env, init_states[ep], PROMPT, vanilla_policy_fn, episode_idx=ep,
#     )
#     tag = "SUCCESS" if success else "FAILURE"
#     mp4 = BASELINE_DIR / f"ep{ep:02d}--{tag}.mp4"
#     save_video(frames, mp4)
#     dt = time.time() - t0
#     print(f"baseline ep {ep:2d}: {tag:7s}  steps={env_steps:4d}  {dt:6.1f}s  -> {mp4.name}",
#           flush=True)
#     baseline_results.append({
#         "episode":     ep,
#         "success":     success,
#         "env_steps":   env_steps,
#         "wall_time_s": dt,
#         "video_path":  str(mp4),
#     })

# (BASELINE_DIR / "results.json").write_text(json.dumps(baseline_results, indent=2))
# n_succ_b = sum(r["success"] for r in baseline_results)
# print(f"baseline total {time.time()-_BASE_T0:.1f}s; "
#       f"{n_succ_b}/{len(baseline_results)} succeeded "
#       f"({100*n_succ_b/len(baseline_results):.0f}%)")
# print(f"saved -> {BASELINE_DIR.resolve()}")

# env.close()


## 4.2 Per-control diagnostic (aggregated over the whole rollout)

In [ ]:
if u_norm_log:
    log = torch.tensor(u_norm_log)
    # rows: (sel_idx, l_in, ||u||, alpha).  l_in == -1 marks cross-step entries.
    intra = log[log[:, 1] != -1]
    step  = log[log[:, 1] == -1]
    print(f"u_norm_log entries: intra={len(intra)}, cross-step={len(step)}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    if len(intra):
        u_mean = torch.zeros(T_diff, L - 1)
        u_cnt  = torch.zeros(T_diff, L - 1)
        for s, l, un, _al in intra.tolist():
            u_mean[int(s), int(l)] += un
            u_cnt[int(s),  int(l)] += 1
        u_mean = (u_mean / u_cnt.clamp(min=1)).numpy()
        im0 = axes[0].imshow(u_mean, aspect="auto", origin="lower", cmap="magma")
        axes[0].set_title(r"mean within-step $\Vert\tilde u\Vert$ over all chunks")
        axes[0].set_xlabel(r"$l_{\rm in}$"); axes[0].set_ylabel(r"$t_{\rm sel}$")
        plt.colorbar(im0, ax=axes[0])

    if len(step) and T_diff > 1:
        u_step = torch.zeros(T_diff - 1)
        c_step = torch.zeros(T_diff - 1)
        for s, _l, un, _al in step.tolist():
            if 0 <= int(s) < T_diff - 1:
                u_step[int(s)] += un
                c_step[int(s)] += 1
        u_step = (u_step / c_step.clamp(min=1)).numpy()
        axes[1].plot(range(T_diff - 1), u_step, "o-", color="C3")
        axes[1].set_title(r"mean cross-step $\Vert\tilde u\Vert$")
        axes[1].set_xlabel(r"$t_{\rm sel}$")
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, "no cross-step (T_sel <= 1)",
                     ha="center", va="center", transform=axes[1].transAxes)
    plt.tight_layout(); plt.show()
else:
    print("u_norm_log is empty; run the steered rollout cell above first.")
